In [ ]:
# Top level imports
import os
import json
from collections import defaultdict
from typing import Literal
from pprint import pprint
from pathlib import Path

import pandas as pd
import jsonlines
import lancedb
import pyarrow as pa
import pytrec_eval
import numpy as np
from dotenv import load_dotenv
from tqdm import tqdm
from beir.datasets.data_loader import GenericDataLoader
from openai import OpenAI

load_dotenv(override=True)

In [ ]:
# Configurations
dataset_basedir = "datasets/beir/datasets"
datasets = [
    "scifact", "trec-covid", "nfcorpus", "fiqa", "arguana", 
    "scidocs", "quora", "dbpedia-entity", "fever", "climate-fever",
    "hotpotqa", "nq", "msmarco"  # Warning: msmarco is very large!
]
batch_size = 4

dataset = "scifact"
embedding_model = "text-embedding-3-small"
table = None 
client = OpenAI()

uri = "labcedb_index"
db = lancedb.connect(uri)

In [ ]:
# Data Loaders

def batch_loader(dataset_path, batch_size):
    '''Load jsonlines dataset in batches.'''
    batch = []
    with jsonlines.open(dataset_path, 'r') as reader: 
        for line in reader:
            batch.append(line)
            if len(batch) == batch_size:
                yield batch
                batch = []

        if len(batch):
            yield batch

def test_batch_loader(batch_size=4):
    '''Test a single batch'''
    loader = batch_loader("datasets/beir/datasets/scifact/corpus.jsonl", batch_size=batch_size)

    for batch in loader: 
        print(f"TEST: 1 batch from the batch loader, batch_size={batch_size}")
        for line in batch:
            print(line)
        break

test_batch_loader()

In [5]:
# Chunk - batch + encode + index + vectorDB

schema = pa.schema([
    pa.field("id", pa.string(), nullable=False),
    pa.field("full_text", pa.string(), nullable=True),
    pa.field("vector", pa.list_(pa.float32(), 1536), nullable=False) # Fixes the 'None' vector issue
])

def create_and_store_embeddings(dataset):
    '''Create doc embeddings with OpenAI client. Store in vector DB with flat index.'''
    loader = batch_loader(os.path.join(dataset_basedir, dataset, "corpus.jsonl"), batch_size=batch_size)

    for batch in tqdm(loader):
        texts_to_encode = [x["title"] + " - " + x["text"] for x in batch]
        response = client.embeddings.create(
            model=embedding_model,
            input=texts_to_encode
        )
        vectors = [emb.embedding for emb in response.data]

        data_to_insert = [
            {
                "id": x["_id"],
                "full_text": text,
                "vector": v
            }
            for x, text, v in zip(batch, texts_to_encode, vectors)
        ]
        if table is None:
            table = db.create_table("bier_corpus", schema=schema, data=data_to_insert, mode="overwrite")
        else:
            table.add(data_to_insert)

    # break


In [ ]:
# Retrieval
def generate_query_embeddings(query_loader, dataset):
    table = None    

    for batch in tqdm(query_loader):
        q_ids = [x["_id"] for x in batch]
        queries = [x["text"] for x in batch]
        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=queries
        )

        q_vectors = [emb.embedding for emb in response.data]
        data_to_insert = [
                {
                    "id": _id,
                    "full_text": x,
                    "vector": v
                }
                for _id, x, v in zip(q_ids, queries, q_vectors) 
        ]

        if table is None:
            table = db.create_table(f"bier_queries_{dataset}", schema=schema, data=data_to_insert)
        else:
            table.add(data_to_insert)
         

def retrieval(dataset, output_path):
    '''
    Retrieve top-k matches.
    Store tab-separated "query_id   doc_id score".
    '''
    corpus_table = db.open_table("bier_corpus")
    query_table = db.open_table(f"bier_queries_{dataset}")
    print("Corpus size: ", len(corpus_table), " Num queries=", len(query_table))

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, 'w') as fp:
        fp.write(f"query-id\tcorpus-id\tscore\n")

    query_loader = query_table.search().select(['id', 'full_text', 'vector']).to_batches()

    for batch in tqdm(query_loader):
        data_dict = batch.to_pydict()
        q_ids = data_dict['id']
        queries = data_dict['full_text']
        q_vectors = data_dict['vector']

        for _id, q_v in zip(q_ids, q_vectors):
        # for i, q_v in enumerate(q_vectors):
            results = (
                corpus_table.search(q_v)
                .metric("cosine")  # Options: "l2" (default), "cosine", or "dot"
                .limit(10)
                .to_list()
            )
            q_outputs = []
            q_outputs.append([f"{_id}\t{res['id']}\t{1-res["_distance"]}\n" for res in results])

            with open(output_path, 'a+') as fp:
                fp.writelines(q_outputs[0])
                
        # break       


query_loader = batch_loader(os.path.join(dataset_basedir, dataset, "queries.jsonl"), batch_size=batch_size)
output_path = os.path.join("outputs", dataset, "qrels.tsv")

# Check if query tables exists in db
if not f"bier_queries_{dataset}" in db.table_names():
    print(f"Creating query embeddings table")
    generate_query_embeddings(query_loader, dataset)
else:
    print(f"Using existing query embeddings table from db!")

# Retrieve documents against each query    
retrieval(dataset, output_path)


In [ ]:
# Evaluation

def convert_trec_format(path_tsv_data, relevance_type: Literal["int", "float"]):
    '''
    Convert from "query_id  doc_id  score" to trec_eval compatible format.
    '''

    output = defaultdict(dict)

    with open(path_tsv_data, "r") as fp:
        tsv_data = fp.readlines()

    for line in tsv_data[1:]:  # leave the first header line
        q_id, doc_id, score = line.split("\t")
        if relevance_type == "int":
            output[q_id][doc_id] = int(score)
        else:
            output[q_id][doc_id] = float(score)

    return output


qrels = convert_trec_format(
    os.path.join(dataset_basedir, dataset, 'qrels', 'train.tsv'),
    relevance_type="int",
)
runs = convert_trec_format(
    output_path,
    relevance_type="float",
)
metrics = set(
    {
        "map_cut.1",
        "ndcg_cut.1",
        "recall.1",
        "P.1",
        "map_cut.5",
        "ndcg_cut.5",
        "recall.5",
        "P.5",
        "map_cut.10",
        "ndcg_cut.10",
        "recall.10",
        "P.10",
    }
)

# Using pytrec_eval for evaluation
evaluator = pytrec_eval.RelevanceEvaluator(
    qrels,
    metrics,
)
results = evaluator.evaluate(runs)

for metric in sorted(list(metrics)):
    print(f"{metric}={np.mean([x[metric.replace('.', '_')] for x in results.values()])}")
